# **PHASE 9 — BÁO CÁO TỔNG KẾT**

**Cell 9.1: Bảng tổng hợp metrics**

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, balanced_accuracy_score,
                              cohen_kappa_score, confusion_matrix)

BASE = 'D:/DoAnTriTueNhanTao/skin_cancer'
CLASSES = ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC']
MALIGNANT = ['MEL', 'BCC', 'SCC', 'AK']

bin_pred = pd.read_csv(f'{BASE}/results/binary_test_predictions.csv')
mc_pred  = pd.read_csv(f'{BASE}/results/multiclass_test_predictions.csv')

# Binary
y_bin = bin_pred['binary_label'].values
p_bin = bin_pred['pred_binary'].values
prob_bin = bin_pred['prob_malignant'].values

# Multi-class
y_mc = mc_pred['label'].values
p_mc = mc_pred['pred_label'].values
prob_malig_mc = mc_pred[[f'prob_{c}' for c in MALIGNANT]].sum(axis=1).values
y_bin_from_mc = mc_pred['label_name'].apply(lambda x: 1 if x in MALIGNANT else 0).values
p_bin_from_mc = mc_pred['pred_label_name'].apply(lambda x: 1 if x in MALIGNANT else 0).values

summary = pd.DataFrame({
    'Model': ['Binary EfficientNet-B0',
              'Multi-class (8 lớp)',
              'Multi-class → Binary'],
    'Acc': [
        accuracy_score(y_bin, p_bin),
        accuracy_score(y_mc, p_mc),
        accuracy_score(y_bin_from_mc, p_bin_from_mc)
    ],
    'Balanced Acc': [
        balanced_accuracy_score(y_bin, p_bin),
        balanced_accuracy_score(y_mc, p_mc),
        balanced_accuracy_score(y_bin_from_mc, p_bin_from_mc)
    ],
    'Precision': [
        precision_score(y_bin, p_bin),
        np.nan,
        precision_score(y_bin_from_mc, p_bin_from_mc)
    ],
    'Recall': [
        recall_score(y_bin, p_bin),
        np.nan,
        recall_score(y_bin_from_mc, p_bin_from_mc)
    ],
    'F1': [
        f1_score(y_bin, p_bin),
        f1_score(y_mc, p_mc, average='macro'),
        f1_score(y_bin_from_mc, p_bin_from_mc)
    ],
    'AUC / Kappa': [
        roc_auc_score(y_bin, prob_bin),
        cohen_kappa_score(y_mc, p_mc),
        roc_auc_score(y_bin_from_mc, prob_malig_mc)
    ]
})

print("=" * 90)
print("BẢNG TỔNG HỢP KẾT QUẢ — TEST SET")
print("=" * 90)
print(summary.round(4).to_string(index=False))
summary.to_csv(f'{BASE}/results/final_summary.csv', index=False)

# In chi tiết multi-class
print("\n" + "="*90)
print("CHI TIẾT MULTI-CLASS (8 LỚP)")
print("="*90)
print(f"Accuracy (top-1):     {accuracy_score(y_mc, p_mc):.4f}")
print(f"Balanced accuracy:    {balanced_accuracy_score(y_mc, p_mc):.4f}")
print(f"F1-macro:             {f1_score(y_mc, p_mc, average='macro'):.4f}")
print(f"Cohen's Kappa:        {cohen_kappa_score(y_mc, p_mc):.4f}")

# Nhị phân từ multi-class
print("\n" + "="*90)
print("CHI TIẾT NHỊ PHÂN (từ multi-class gộp lớp ác tính)")
print("="*90)
print(f"Accuracy:             {accuracy_score(y_bin_from_mc, p_bin_from_mc):.4f}")
print(f"Recall (sensitivity): {recall_score(y_bin_from_mc, p_bin_from_mc):.4f}  ← quan trọng")
print(f"Precision:            {precision_score(y_bin_from_mc, p_bin_from_mc):.4f}")
print(f"F1:                   {f1_score(y_bin_from_mc, p_bin_from_mc):.4f}")
print(f"AUC-ROC:              {roc_auc_score(y_bin_from_mc, prob_malig_mc):.4f}")

# Confusion
cm = confusion_matrix(y_bin_from_mc, p_bin_from_mc)
tn, fp, fn, tp = cm.ravel()
print(f"\nTN={tn}, FP={fp}, FN={fn}, TP={tp}")
print(f"Bỏ sót ung thư (FN rate): {fn/(fn+tp)*100:.2f}%")
print(f"Báo động giả (FP rate):   {fp/(fp+tn)*100:.2f}%")

BẢNG TỔNG HỢP KẾT QUẢ — TEST SET
                 Model    Acc  Balanced Acc  Precision  Recall     F1  AUC / Kappa
Binary EfficientNet-B0 0.8626        0.8553     0.8056  0.8274 0.8163       0.9366
   Multi-class (8 lớp) 0.8163        0.7815        NaN     NaN 0.7547       0.7305
  Multi-class → Binary 0.8842        0.8740     0.8486  0.8352 0.8418       0.9283

CHI TIẾT MULTI-CLASS (8 LỚP)
Accuracy (top-1):     0.8163
Balanced accuracy:    0.7815
F1-macro:             0.7547
Cohen's Kappa:        0.7305

CHI TIẾT NHỊ PHÂN (từ multi-class gộp lớp ác tính)
Accuracy:             0.8842
Recall (sensitivity): 0.8352  ← quan trọng
Precision:            0.8486
F1:                   0.8418
AUC-ROC:              0.9283

TN=2189, FP=209, FN=231, TP=1171
Bỏ sót ung thư (FN rate): 16.48%
Báo động giả (FP rate):   8.72%


**Cell 9.2: Export bảng LaTeX cho báo cáo**

In [9]:
import os
BASE = r'D:\DoAnTriTueNhanTao\skin_cancer'

# --- LaTeX ---
latex_table = summary.round(4).to_latex(index=False, escape=False)
with open(f'{BASE}/results/final_summary.tex', 'w', encoding='utf-8') as f:
    f.write(latex_table)
print("[OK] Đã lưu LaTeX: final_summary.tex")
print(latex_table)

# --- Markdown ---
with open(f'{BASE}/results/final_summary.md', 'w', encoding='utf-8') as f:
    f.write("# Kết quả tổng hợp — Test set\n\n")
    f.write(summary.round(4).to_markdown(index=False))
print("\n[OK] Đã lưu Markdown: final_summary.md")

[OK] Đã lưu LaTeX: final_summary.tex
\begin{tabular}{lrrrrrr}
\toprule
Model & Acc & Balanced Acc & Precision & Recall & F1 & AUC / Kappa \\
\midrule
Binary EfficientNet-B0 & 0.862600 & 0.855300 & 0.805600 & 0.827400 & 0.816300 & 0.936600 \\
Multi-class (8 lớp) & 0.816300 & 0.781500 & NaN & NaN & 0.754700 & 0.730500 \\
Multi-class → Binary & 0.884200 & 0.874000 & 0.848600 & 0.835200 & 0.841800 & 0.928300 \\
\bottomrule
\end{tabular}


[OK] Đã lưu Markdown: final_summary.md


**Cell 9.3: Kiểm tra toàn bộ artifacts**

In [11]:
import os
BASE = r'D:\DoAnTriTueNhanTao\skin_cancer'

artifacts = [
    # Checkpoints
    'checkpoints/best_binary.pth',
    'checkpoints/best_multiclass.pth',
    # Metadata
    'train.csv', 'val.csv', 'test.csv', 'isic2019_metadata.csv',
    # Results
    'results/binary_history.csv',
    'results/multiclass_history.csv',
    'results/binary_test_predictions.csv',
    'results/multiclass_test_predictions.csv',
    'results/model_comparison.csv',
    'results/final_summary.csv',
    'results/final_summary.tex',
    'results/final_summary.md',
    # Figures
    'figures/class_distribution.png',
    'figures/sample_images.png',
    'figures/binary_learning_curves.png',
    'figures/multiclass_learning_curves.png',
    'figures/binary_test_evaluation.png',
    'figures/multiclass_confusion_matrix.png',
    'figures/multiclass_confusion_matrix_norm.png',
    'figures/multiclass_per_class_f1.png',
    'figures/gradcam_per_class.png',
    'figures/gradcam_errors.png',
]

print("=" * 70)
print("KIỂM TRA ARTIFACTS")
print("=" * 70)
missing = 0
for f in artifacts:
    p = f'{BASE}/{f}'
    if os.path.exists(p):
        sz = os.path.getsize(p)
        szs = f"{sz/1e6:.2f}MB" if sz > 1e5 else f"{sz/1e3:.1f}KB"
        print(f"✓ {f:55s} {szs}")
    else:
        print(f"✗ {f:55s} MISSING")
        missing += 1

print("=" * 70)
print(f"Tổng: {len(artifacts) - missing}/{len(artifacts)} có sẵn")

KIỂM TRA ARTIFACTS
✓ checkpoints/best_binary.pth                             17.65MB
✓ checkpoints/best_multiclass.pth                         17.65MB
✓ train.csv                                               1.97MB
✓ val.csv                                                 0.42MB
✓ test.csv                                                0.42MB
✓ isic2019_metadata.csv                                   2.76MB
✓ results/binary_history.csv                              2.0KB
✓ results/multiclass_history.csv                          2.2KB
✓ results/binary_test_predictions.csv                     0.35MB
✓ results/multiclass_test_predictions.csv                 0.66MB
✓ results/model_comparison.csv                            0.3KB
✓ results/final_summary.csv                               0.4KB
✓ results/final_summary.tex                               0.4KB
✓ results/final_summary.md                                0.6KB
✓ figures/class_distribution.png                          38.4KB
✓ figures/